The dataset used : https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia/data

All images are in jpeg format. As we can't use libraries to deal with this format, we have to use pgm format, which can be turned into numpy ndarray for scikit-learn.

Search for : *Application of the AI System for Pneumonia Detection Using Chest X-Ray Images* in http://www.cell.com/cell/fulltext/S0092-8674(18)30154-5

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize

As Scikit-learn only deals with mathematical objects (in this case numpy ndarray), we need to do some operations on our dataset.
Our images are in .jpeg format. We use scikit-image library to convert them into numpy ndarray.

Simple bash script using ImageMagik to resize images. The reason we do it in bash is bash is much faster than python to do this kind of operation. 

```bash
find archive/chest_xray/train -type f -name "*.jpeg" -exec mogrify -strip -resize 64x64! {} +
```


Tests were made using a resolution of 128x128. The results were sligthly better (+ ~0.3% of precision). But the execution time was almost 3 times longer. So we decided to keep 64x64. 

In [2]:
input_dir  = 'archive/chest_xray/train'
categories = ['NORMAL', 'PNEUMONIA']

data   = []  # images as numerical data
labels = []  # categories corresponding to each images

for category_idx, category in enumerate(categories):
    folder: str = os.path.join(input_dir, category)
    for file in os.scandir(folder):
        img_path = file.path
        img      = imread(img_path)
        if img.ndim == 3:
            img = rgb2gray(img)  # conversion to shade of gray for optimisation
        data.append(img)
        labels.append(category_idx)

data   = np.array(data, dtype=np.float32)
data   = data.reshape(len(data), -1)
labels = np.array(labels)

# We save the numpy array as .npy so we can use our dataset without restarting this block of code
np.save("data.npy", data) 
np.save("labels.npy", labels)

In [3]:
# Load the data from the files created above. This avoid recreating the dataset every time.
data   = np.load("data.npy")
labels = np.load("labels.npy")

For a proper training, we need to split the dataset.\
The idea is to create a "test" sample of the dataset. If we give the all dataset to the model and then evaluate on the same datas, will shall get a 100% precision. But it won't be accurate. So we make sure to evaluate the model on data it never seen.

In [4]:
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

We create a "results" tab will store the scores of the different metrics for each model.

In [5]:
results = []

The pipeline allows us to put a "scaller". It standardize the data before passing it to the models.

In [6]:
pipeline = Pipeline([
    ('scalify', StandardScaler()),
    ('model', DummyClassifier())
])

To test and evaluate the different models, we use a GridSearch. It will allows us to test the same model with different parameters and store the results of the metrics in the "results" array.

In [7]:
def create_grid_search(model_name, classifier, parameters) -> None:

    pipeline.set_params(model=classifier)

    grid_search = GridSearchCV(
        pipeline,
        param_grid=parameters,
        cv=4,
        n_jobs=-1,
        scoring='accuracy',
        verbose=2
    )
    grid_search.fit(x_train, y_train)

    best_estimator = grid_search.best_estimator_
    y_prediction   = best_estimator.predict(x_test)

    score : float      = accuracy_score(y_prediction, y_test)
    report: str | dict = classification_report(y_test, y_prediction, target_names=categories)

    print(f"Metrics for {model_name}:")
    print(f"Best parameters: {grid_search.best_params_}")
    print(report)

    results.append({
        'model': model_name,
        'accuracy': score,
        'best_params': grid_search.best_params_,
        'estimator': best_estimator
    })
    print('{}% of samples were correctly classified'.format(str(score * 100)))

# K-Nearest Neighbors

The K-Nearest Neighbors algorithm classifies an image by looking at the K most similar images in the training set. It measures distance between images (feature vectors)

In [8]:
KNeighborsClassifier_parameters = [{'model__n_neighbors' : [3, 5, 10, 15, 20]}]
create_grid_search("KNeighborsClassifier", KNeighborsClassifier(), KNeighborsClassifier_parameters)

Fitting 4 folds for each of 5 candidates, totalling 20 fits
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ..............................model__n_neighbors=10; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ..............................model__n_neighbors=10; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.3min
[CV] END ...............................model__n_neighbors=5; total time= 1.3min
[CV] END ...............................model__n_neighbors=3; total time= 1.4min
[CV] END ..............................model__n_neighbors=10; total time= 1.4min
[CV] END ..............................model__n_n

# Metrics for KNeighborsClassifier:

**With 64x64**:

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.93      | 0.81   | 0.87     | 315     |
| PNEUMONIA | 0.93      | 0.98   | 0.95     | 854     |

Accuracy: 92.89%

Macro Avg:    Precision = 0.93 | Recall = 0.89 | F1-score = 0.91\
Weighted Avg: Precision = 0.93 | Recall = 0.93 | F1-score = 0.93

**With 128x128**

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.95      | 0.79   | 0.87     | 315     |
| PNEUMONIA | 0.93      | 0.98   | 0.96     | 854     |

Accuracy: 93.00%

Macro Avg:    Precision = 0.94 | Recall = 0.89 | F1-score = 0.91\
Weighted Avg: Precision = 0.93 | Recall = 0.93 | F1-score = 0.93

# DecisionTreeClassifier

A Decision Tree splits the data step by step using simple rules like "Is pixel X > value" -> yes / no 

In [9]:
DecisionTreeClassifier_parameters = [{
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}]
create_grid_search("DecisionTreeClassifier", DecisionTreeClassifier(), DecisionTreeClassifier_parameters)

Fitting 4 folds for each of 36 candidates, totalling 144 fits
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=2; total time=   8.4s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=10; total time=   8.6s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=2; total time=   8.6s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=10; total time=   8.7s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=10; total time=   8.7s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=2; total time=   8.7s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=   8.9s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=   9.0s
[CV] END model__max_depth=5, model__min_samples_leaf=1, model__min_samples_split=5; total time=   9.0s
[CV] END

# Metrics for DecisionTreeClassifier:

**With 64x64**:

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.81      | 0.79   | 0.80     | 315     |
| PNEUMONIA | 0.92      | 0.93   | 0.93     | 854     |

Accuracy: 87.85%

Macro Avg:    Precision = 0.87 | Recall = 0.79 | F1-score = 0.80\
Weighted Avg: Precision = 0.89 | Recall = 0.89 | F1-score = 0.89


**With 128x128**

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.81      | 0.80   | 0.80     | 315     |
| PNEUMONIA | 0.93      | 0.93   | 0.93     | 854     |

Accuracy: 89.00%

Macro Avg:    Precision = 0.87 | Recall = 0.86 | F1-score = 0.86\
Weighted Avg: Precision = 0.89 | Recall = 0.89 | F1-score = 0.89

# RandomForestClassifier

A Random Forest is a collection of many decision trees. Each tree is trained on a random subset of the data.

In [10]:
RandomForestClassifier_parameters = [{
    'model__max_depth': [10, 30, None],
    'model__min_samples_leaf': [1, 2, 4],
    'model__min_samples_split': [2, 5, 10],
    'model__n_estimators': [100, 300]
}]
create_grid_search("RandomForestClassifier", RandomForestClassifier(), RandomForestClassifier_parameters)

Fitting 4 folds for each of 54 candidates, totalling 216 fits
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  18.2s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  18.4s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  18.4s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  18.4s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  18.8s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=100; total time=  18.8s
[CV] END model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=  19.0s
[CV] END model__max_depth=10, model

# Metrics for RandomForestClassifier:

**With 64x64**:

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.94      | 0.87   | 0.90     | 315     |
| PNEUMONIA | 0.95      | 0.98   | 0.97     | 854     |

Accuracy: 94.95%

Macro Avg:    Precision = 0.95 | Recall = 0.92 | F1-score = 0.93\
Weighted Avg: Precision = 0.95 | Recall = 0.95 | F1-score = 0.95


**With 128x128**

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.96      | 0.87   | 0.91     | 315     |
| PNEUMONIA | 0.95      | 0.99   | 0.97     | 854     |

Accuracy: 95.46%

Macro Avg:    Precision = 0.96 | Recall = 0.93 | F1-score = 0.94\
Weighted Avg: Precision = 0.95 | Recall = 0.95 | F1-score = 0.95

# Support Vector Classifier (SVC)

The SVC tries to find the best boundary that separates the classes.

In [11]:
svc_parameters = [{'model__gamma': [0.01, 0.001, 0.0001], 'model__C': [1, 10, 100, 1000]}]
create_grid_search("SVC", SVC(), svc_parameters)

Fitting 4 folds for each of 12 candidates, totalling 48 fits
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.2min
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.2min
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.2min
[CV] END ....................model__C=1, model__gamma=0.0001; total time= 1.2min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 3.9min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 3.9min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 4.2min
[CV] END .....................model__C=1, model__gamma=0.001; total time= 4.2min
[CV] END ......................model__C=1, model__gamma=0.01; total time= 4.7min
[CV] END ......................model__C=1, model__gamma=0.01; total time= 4.9min
[CV] END ......................model__C=1, model__gamma=0.01; total time= 4.9min
[CV] END ......................model__C=1, model

# Metrics for SVC:

**With 64x64**:

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.93      | 0.93   | 0.93     | 315     |
| PNEUMONIA | 0.98      | 0.97   | 0.97     | 854     |

Accuracy: 96.32%

Macro Avg:    Precision = 0.95 | Recall = 0.95 | F1-score = 0.95\
Weighted Avg: Precision = 0.96 | Recall = 0.96 | F1-score = 0.96


**With 128x128**

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.94      | 0.94   | 0.94     | 315     |
| PNEUMONIA | 0.98      | 0.98   | 0.98     | 854     |

Accuracy: 96.66%

Macro Avg:    Precision = 0.96 | Recall = 0.96 | F1-score = 0.96\
Weighted Avg: Precision = 0.97 | Recall = 0.97 | F1-score = 0.97

We calculate the best model based on the results from the "results" array...

# Multi-Layer Perceptron (MLP)

To push experimentations, we added another kind of model. Which use a simple neural network.

In this network each neuron:
- takes inputs
- applies weights
- passes through an activation function

During training the model adjusts weights to minimize error thank to a *loss function*.

**Loss Function**

A loss function measures how far the model’s predictions are from the true labels.
During training, the model iteratively updates its parameters to minimize this loss.

In this project, the MLPClassifier uses the log-loss (cross-entropy) function.
This loss evaluates the difference between predicted probabilities and actual classes.

It penalizes incorrect and overconfident predictions more strongly, encouraging the model to produce accurate and well-calibrated probabilities.

Additionally, regularization (controlled by the parameter alpha) is applied to prevent overfitting by penalizing overly complex models.

In [12]:
mlp_parameters = [{
    'model__hidden_layer_sizes': [(50,), (100,), (50,50)],
    'model__activation': ['relu', 'tanh'],
    'model__solver': ['adam'],
    'model__alpha': [0.0001, 0.001]
}]
create_grid_search("MLP", MLPClassifier(max_iter=500, early_stopping=True), mlp_parameters) # the early_stopping parameter reduces training time by stoping it when no improvement appears anymore

Fitting 4 folds for each of 12 candidates, totalling 48 fits
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50, 50), model__solver=adam; total time=  33.8s
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50, 50), model__solver=adam; total time=  48.1s
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50,), model__solver=adam; total time=  58.0s
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50, 50), model__solver=adam; total time= 1.0min
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50,), model__solver=adam; total time= 1.1min
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50, 50), model__solver=adam; total time= 1.2min
[CV] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(50,), model__solver=adam; total time= 1.3min
[CV] END model__activation=relu, model__alpha

# Metrics for MLP Classifer:

**With 64x64**:

| Class     | Precision | Recall | F1-score | Support |
| --------- | --------- | ------ | -------- | ------- |
| NORMAL    | 0.85      | 0.91   | 0.88     | 315     |
| PNEUMONIA | 0.97      | 0.94   | 0.95     | 854     |

Accuracy: 93.32%

Macro Avg:    Precision = 0.91 | Recall = 0.93 | F1-score = 0.92\
Weighted Avg: Precision = 0.94 | Recall = 0.93 | F1-score = 0.93

Since the neural network model is slower we didn't test with 128x128.

In [13]:
results_sorted = sorted(results, key=lambda x: x['accuracy'], reverse=True)

for res in results_sorted:
    print(f"Best model : {res['model']:20} | Accuracy: {res['accuracy']:.4f}")

Best model : SVC                  | Accuracy: 0.9581
Best model : RandomForestClassifier | Accuracy: 0.9384
Best model : MLP                  | Accuracy: 0.9333
Best model : KNeighborsClassifier | Accuracy: 0.9290
Best model : DecisionTreeClassifier | Accuracy: 0.8785


...Then we dump (save) the best model to re-use it later.

In [14]:
best_result = max(results, key=lambda x: x['accuracy'])
best_model = best_result['estimator']

joblib.dump(best_model, "CVRIE_PNEUMONIA_RECOGNITION.pkl")

['CVRIE_PNEUMONIA_RECOGNITION.pkl']